# IMA205 Challenge: White Blood Cell Classification

This notebook presents a complete pipeline for classifying 13 types of white blood cells (WBCs) from microscopy images. The dataset is heavily imbalanced — SNE accounts for ~45% of training samples while PLY has only 11.

Two complementary approaches are combined:
1. **Traditional ML**: nucleus-cropped images → hand-crafted features (color, shape, texture, HOG) → XGBoost with oversampling + augmentation
2. **Deep Learning**: fine-tuned CNN/ViT backbones (ResNet-50, EfficientNet-B3, ConvNeXt-Large) trained with Focal Loss + k-fold cross-validation, ensembled with 8-view TTA at inference

Class counts (train): BA=415, BL=2012, BNE=391, EO=861, LY=8101, MMY=360, MO=2746, MY=441, PC=68, PLY=11, PMY=114, SNE=13015, VLY=366

## Part 1 — Traditional ML Pipeline (XGBoost)

### 1.1 Imports and Configuration

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image

import cv2
import matplotlib.pyplot as plt
import seaborn as sns

from skimage.transform import resize
from skimage.feature import hog, local_binary_pattern
from skimage.color import rgb2hsv, rgb2gray, rgb2lab
from skimage.measure import moments_hu, label as sk_label, regionprops
from skimage.morphology import binary_opening, binary_closing, disk
from skimage.filters import threshold_otsu
from scipy.stats import skew, kurtosis
from scipy.ndimage import binary_fill_holes

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight
from joblib import Parallel, delayed
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=UserWarning)

ROOT_DIR = "./"
TRAIN_DIR = os.path.join(ROOT_DIR, "train")
TEST_DIR  = os.path.join(ROOT_DIR, "test")
TRAIN_META = os.path.join(ROOT_DIR, "train_metadata.csv")
TEST_META  = os.path.join(ROOT_DIR, "test_metadata.csv")

IMG_SIZE = 128
AUG_SEED = 42
N_AUG = 10                       # number of augmentation views per image
OVERSAMPLE_MIN_PER_CLASS = 600   # upsample minority classes to this count

### 1.2 Data Loading

In [ ]:
train_df = pd.read_csv(TRAIN_META)
test_df  = pd.read_csv(TEST_META)

le = LabelEncoder()
y_train = le.fit_transform(train_df["label"])

print("Training samples:", len(train_df), "| Classes:", len(le.classes_))
print(train_df["label"].value_counts())

### 1.3 WBC Nucleus Cropping

Each image contains a single WBC on a light background. We locate the nucleus using HSV thresholding (high saturation + low brightness), find the connected component closest to the image center, and crop a fixed-size square around its centroid. If the crop would fall outside the image bounds, we shift the window rather than padding with zeros.

In [ ]:
def apply_wbc_crop_robust(img_rgb: np.ndarray, fixed_size: int = 220) -> np.ndarray:
    h, w = img_rgb.shape[:2]
    img_hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)

    mask = ((img_hsv[:, :, 1] > 50) & (img_hsv[:, :, 2] < 190)).astype(np.uint8) * 255

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    num, labels, stats, centroids = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if num <= 1:
        return img_rgb

    img_center = np.array([w / 2, h / 2])
    best_idx, min_dist = -1, float('inf')
    for i in range(1, num):
        if stats[i, cv2.CC_STAT_AREA] < 400:
            continue
        dist = np.linalg.norm(centroids[i] - img_center)
        if dist < min_dist:
            min_dist = dist
            best_idx = i

    if best_idx == -1:
        return img_rgb

    cx, cy = int(centroids[best_idx][0]), int(centroids[best_idx][1])
    half_s = fixed_size // 2
    x1, x2 = cx - half_s, cx + half_s
    y1, y2 = cy - half_s, cy + half_s

    if x1 < 0:   x2 -= x1; x1 = 0
    if y1 < 0:   y2 -= y1; y1 = 0
    if x2 > w:   x1 -= (x2 - w); x2 = w
    if y2 > h:   y1 -= (y2 - h); y2 = h
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)

    return img_rgb[y1:y2, x1:x2]

In [ ]:
# Visualize the crop on a random training image
sample_row = train_df.sample(1).iloc[0]
img_rgb = np.array(Image.open(os.path.join(TRAIN_DIR, sample_row['ID'])).convert("RGB"))
cropped  = apply_wbc_crop_robust(img_rgb, fixed_size=220)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img_rgb);  axes[0].set_title(f"Original  {img_rgb.shape[:2]}  [{sample_row['label']}]")
axes[1].imshow(cropped);  axes[1].set_title(f"Cropped  {cropped.shape[:2]}")
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

### 1.4 Feature Extraction (V2)

We use a two-level Otsu segmentation to separate the cell from background, then nucleus from cytoplasm. Features are drawn from four sources:
- **Nucleus**: shape (area, circularity, eccentricity…), HSV color stats, HSV histogram, LBP texture
- **Cytoplasm**: same color/texture descriptors as nucleus
- **N/C ratio**: nucleus-to-cytoplasm area ratio and related scalars
- **Global**: full-image HSV/Lab histograms, color statistics, HOG, Hu moments

Total: ~718 dimensions per image.

In [ ]:
# ── Two-level segmentation ──────────────────────────────────────────────────

def segment_cell_and_nucleus(img_rgb):
    img_hsv = rgb2hsv(img_rgb)
    img_lab = rgb2lab(img_rgb)
    v    = img_hsv[:, :, 2]
    b_ch = img_lab[:, :, 2]
    total = IMG_SIZE * IMG_SIZE

    try:
        thresh_v1 = threshold_otsu(v)
        thresh_b  = threshold_otsu(b_ch)
    except Exception:
        thresh_v1, thresh_b = 0.6, 0

    cell_mask = (v < thresh_v1) & (b_ch < thresh_b)
    cell_mask = binary_fill_holes(binary_opening(binary_closing(cell_mask, disk(2)), disk(1)))

    if cell_mask.sum() > 0.30 * total:
        cell_mask = binary_fill_holes(binary_opening(
            binary_closing((v < thresh_v1 * 0.8) & (b_ch < thresh_b), disk(2)), disk(1)))
    if cell_mask.sum() < 0.02 * total:
        cell_mask = binary_fill_holes(binary_opening(
            binary_closing(v < thresh_v1, disk(2)), disk(1)))

    if cell_mask.sum() > 50:
        v_in_cell = v[cell_mask]
        try:
            thresh_v2 = threshold_otsu(v_in_cell)
        except Exception:
            thresh_v2 = np.median(v_in_cell)
        nucleus_mask = binary_fill_holes(binary_closing(cell_mask & (v < thresh_v2), disk(1)))
    else:
        nucleus_mask = cell_mask

    cytoplasm_mask = cell_mask & (~nucleus_mask)
    return cell_mask, nucleus_mask, cytoplasm_mask


# ── Per-region feature helpers ──────────────────────────────────────────────

def extract_nucleus_shape_features(mask):
    regions = regionprops(sk_label(mask.astype(int)))
    if not regions:
        return np.zeros(6)
    r = max(regions, key=lambda x: x.area)
    total = IMG_SIZE * IMG_SIZE
    return np.array([
        r.area / total,
        r.perimeter / (4 * IMG_SIZE),
        (4 * np.pi * r.area) / (r.perimeter ** 2 + 1e-7),
        r.eccentricity,
        r.solidity,
        r.extent,
    ])

def extract_roi_color_stats(img_rgb, mask):
    if mask.sum() < 10:
        return np.zeros(24)
    stats = []
    for img_space in [rgb2hsv(img_rgb), rgb2lab(img_rgb)]:
        for ch in range(3):
            px = img_space[:, :, ch][mask]
            stats += [
                np.mean(px), np.std(px),
                float(skew(px))     if len(px) > 2 else 0.0,
                float(kurtosis(px)) if len(px) > 2 else 0.0,
            ]
    return np.array(stats)

def extract_roi_hsv_histogram(img_rgb, mask, h_bins=16, s_bins=16, v_bins=16):
    if mask.sum() < 10:
        return np.zeros(h_bins + s_bins + v_bins)
    img_hsv = rgb2hsv(img_rgb)
    hists = []
    for ch, bins in zip(range(3), [h_bins, s_bins, v_bins]):
        h = np.histogram(img_hsv[:, :, ch][mask], bins=bins, range=(0, 1))[0].astype(float)
        hists.append(h / (h.sum() + 1e-7))
    return np.concatenate(hists)

def extract_roi_lbp(img_rgb, mask, radius=1, n_points=8):
    img_uint8 = (rgb2gray(img_rgb) * 255).astype(np.uint8)
    lbp = local_binary_pattern(img_uint8, n_points, radius, method='uniform')
    if mask.sum() < 10:
        return np.zeros(n_points + 2)
    hist, _ = np.histogram(lbp[mask], bins=np.arange(0, n_points + 3), density=True)
    return hist

def extract_nc_ratio_features(cell_mask, nucleus_mask, cytoplasm_mask):
    cell_area = cell_mask.sum()
    if cell_area < 10:
        return np.zeros(3)
    nuc_area  = nucleus_mask.sum()
    cyto_area = cytoplasm_mask.sum()
    return np.array([nuc_area / cell_area, cyto_area / cell_area, nuc_area / (cyto_area + 1)])


# ── Global feature helpers ──────────────────────────────────────────────────

def extract_hsv_histogram(img_rgb, h_bins=30, s_bins=32, v_bins=32):
    img_hsv = rgb2hsv(img_rgb)
    hists = []
    for ch, bins in zip(range(3), [h_bins, s_bins, v_bins]):
        h = np.histogram(img_hsv[:, :, ch], bins=bins, range=(0, 1))[0].astype(float)
        hists.append(h / (h.sum() + 1e-7))
    return np.concatenate(hists)

def extract_lab_histogram(img_rgb, l_bins=32, a_bins=32, b_bins=32):
    img_lab = rgb2lab(img_rgb)
    ranges  = [(0, 100), (-128, 127), (-128, 127)]
    bins_   = [l_bins, a_bins, b_bins]
    hists = []
    for ch, (lo, hi), nb in zip(range(3), ranges, bins_):
        h = np.histogram(img_lab[:, :, ch], bins=nb, range=(lo, hi))[0].astype(float)
        hists.append(h / (h.sum() + 1e-7))
    return np.concatenate(hists)

def extract_color_statistics(img_rgb):
    img_hsv = rgb2hsv(img_rgb)
    stats = []
    for img_cs in [img_rgb, img_hsv]:
        for ch in range(3):
            px = img_cs[:, :, ch].ravel()
            stats += [np.mean(px), np.std(px), float(skew(px)), float(kurtosis(px))]
    return np.array(stats)

def extract_hog_features(img_rgb):
    img_gray = resize(rgb2gray(img_rgb), (IMG_SIZE, IMG_SIZE), anti_aliasing=True)
    return hog(img_gray, orientations=9, pixels_per_cell=(8, 8),
               cells_per_block=(2, 2), block_norm='L2-Hys', feature_vector=True)

def extract_hu_moments(img_rgb):
    img_gray = resize(rgb2gray(img_rgb), (IMG_SIZE, IMG_SIZE), anti_aliasing=True)
    hu = moments_hu(img_gray)
    return -np.sign(hu) * np.log10(np.abs(hu) + 1e-10)


# ── Top-level feature function ───────────────────────────────────────────────

def extract_all_features_v2(img_rgb):
    """Input: IMG_SIZE x IMG_SIZE x 3 float64 in [0, 1]. Returns ~718-d vector."""
    cell_mask, nucleus_mask, cytoplasm_mask = segment_cell_and_nucleus(img_rgb)

    nuc_shape    = extract_nucleus_shape_features(nucleus_mask)         # 6
    nuc_color    = extract_roi_color_stats(img_rgb, nucleus_mask)       # 24
    nuc_hsv_hist = extract_roi_hsv_histogram(img_rgb, nucleus_mask)     # 48
    nuc_lbp      = extract_roi_lbp(img_rgb, nucleus_mask)              # 10

    cyto_color    = extract_roi_color_stats(img_rgb, cytoplasm_mask)    # 24
    cyto_hsv_hist = extract_roi_hsv_histogram(img_rgb, cytoplasm_mask)  # 48
    cyto_lbp      = extract_roi_lbp(img_rgb, cytoplasm_mask)           # 10

    nc_ratio = extract_nc_ratio_features(cell_mask, nucleus_mask, cytoplasm_mask)  # 3

    g_hsv   = extract_hsv_histogram(img_rgb)    # 94
    g_lab   = extract_lab_histogram(img_rgb)     # 96
    g_stats = extract_color_statistics(img_rgb)  # 24
    g_hog   = extract_hog_features(img_rgb)      # 324
    g_hu    = extract_hu_moments(img_rgb)         # 7

    return np.concatenate([
        nuc_shape, nuc_color, nuc_hsv_hist, nuc_lbp,
        cyto_color, cyto_hsv_hist, cyto_lbp,
        nc_ratio,
        g_hsv, g_lab, g_stats, g_hog, g_hu,
    ])

### 1.5 Data Augmentation and Oversampling

To combat class imbalance we do two things:
- **Oversampling**: replicate training rows until every class reaches `OVERSAMPLE_MIN_PER_CLASS` samples
- **Augmentation**: for each (possibly duplicated) image, generate `N_AUG` views at the cropped-but-not-yet-resized stage

Augmentations include rotation, flips, HSV jitter, brightness/contrast shifts, Gaussian blur, noise, gamma correction, and translation — each assigned an integer `aug_id`. `aug_id=0` always returns the original image.

In [ ]:
def apply_aug_rgb(img_rgb_u8: np.ndarray, aug_id: int, rng: np.random.Generator) -> np.ndarray:
    h, w = img_rgb_u8.shape[:2]
    center = (w / 2.0, h / 2.0)

    if aug_id == 0:
        return img_rgb_u8.copy()
    if aug_id == 1:
        M = cv2.getRotationMatrix2D(center, float(rng.uniform(-8.0, 8.0)), 1.0)
        return cv2.warpAffine(img_rgb_u8, M, (w, h), borderMode=cv2.BORDER_REFLECT_101)
    if aug_id == 2:
        return cv2.flip(img_rgb_u8, 1)
    if aug_id == 3:
        return cv2.rotate(img_rgb_u8, cv2.ROTATE_90_CLOCKWISE)
    if aug_id == 4:
        hsv = cv2.cvtColor(img_rgb_u8, cv2.COLOR_RGB2HSV).astype(np.float32)
        hsv[:, :, 0] = (hsv[:, :, 0] + rng.uniform(-6.0, 6.0)) % 180.0
        hsv[:, :, 1] = np.clip(hsv[:, :, 1] * rng.uniform(0.90, 1.10), 0, 255)
        hsv[:, :, 2] = np.clip(hsv[:, :, 2] * rng.uniform(0.90, 1.10), 0, 255)
        return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
    if aug_id == 5:
        return cv2.flip(img_rgb_u8, 0)
    if aug_id == 6:
        return cv2.convertScaleAbs(img_rgb_u8,
                                   alpha=rng.uniform(0.8, 1.2),
                                   beta=rng.uniform(-20, 20))
    if aug_id == 7:
        ksize = int(rng.choice([3, 5]))
        return cv2.GaussianBlur(img_rgb_u8, (ksize, ksize), 0)
    if aug_id == 8:
        noise = rng.normal(0, 5, img_rgb_u8.shape).astype(np.int16)
        return np.clip(img_rgb_u8.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    if aug_id == 9:
        gamma = rng.uniform(0.7, 1.3)
        table = np.array([((i / 255.0) ** (1.0 / gamma)) * 255
                          for i in range(256)]).astype(np.uint8)
        return cv2.LUT(img_rgb_u8, table)
    if aug_id == 10:
        M = np.float32([[1, 0, rng.uniform(-10, 10)], [0, 1, rng.uniform(-10, 10)]])
        return cv2.warpAffine(img_rgb_u8, M, (w, h), borderMode=cv2.BORDER_REFLECT_101)

    return img_rgb_u8.copy()


def enhance_resolution_sense(img_u8: np.ndarray) -> np.ndarray:
    """CLAHE on the L channel in LAB space to sharpen local contrast."""
    lab = cv2.cvtColor(img_u8, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    cl = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8)).apply(l)
    return cv2.cvtColor(cv2.merge((cl, a, b)), cv2.COLOR_LAB2RGB)


def oversample_train_df(df: pd.DataFrame,
                        min_per_class,
                        rng: np.random.Generator) -> pd.DataFrame:
    """Repeat rows with replacement until every class has min_per_class samples."""
    if min_per_class is None:
        return df.copy().reset_index(drop=True)
    parts = []
    for lab in df["label"].unique():
        sub = df[df["label"] == lab]
        parts.append(sub)
        if len(sub) < min_per_class:
            extra = sub.sample(n=min_per_class - len(sub), replace=True,
                               random_state=int(rng.integers(1_000_000_000)))
            parts.append(extra)
    return pd.concat(parts, ignore_index=True)

### 1.6 Parallel Feature Extraction

Each row goes through: crop → optional augmentation → CLAHE enhancement → Lanczos resize to `IMG_SIZE` → feature extraction. Training rows are replicated `N_AUG` times (one per augmentation view); validation/test rows use only the original image. We parallelize over images using `joblib`.

In [ ]:
def process_single_row_parallel(row, image_dir, is_train, n_aug, rng_seed):
    rng = np.random.default_rng(rng_seed)
    img = np.array(Image.open(os.path.join(image_dir, row["ID"])).convert("RGB"))
    img_cropped = apply_wbc_crop_robust(img, fixed_size=220)

    y_label = row["label"] if "label" in row else None
    result_feats, result_labels = [], []

    for aug_id in range(n_aug if is_train else 1):
        aug_u8      = apply_aug_rgb(img_cropped, aug_id, rng)
        img_enh     = enhance_resolution_sense(aug_u8)
        img_resized = cv2.resize(img_enh, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LANCZOS4)
        feat        = extract_all_features_v2(img_resized.astype(np.float64) / 255.0)
        result_feats.append(feat)
        if y_label is not None:
            result_labels.append(y_label)

    return result_feats, result_labels


def extract_features_parallel(df, image_dir, is_train=True, n_aug=5, seed=42):
    res = Parallel(n_jobs=-1)(
        delayed(process_single_row_parallel)(row, image_dir, is_train, n_aug, seed + i)
        for i, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc="Feature extraction"))
    )
    all_feats, all_labels = [], []
    for feats, labels in res:
        all_feats.extend(feats)
        all_labels.extend(labels)

    X = np.array(all_feats)
    y = le.transform(all_labels) if all_labels else None
    return X, y

### 1.7 XGBoost — 5-Fold Cross Validation

Within each fold:
1. Oversample the training split to 600 samples per class
2. Extract features with 10-view augmentation (training) / no augmentation (validation)
3. Standardize features and replace NaNs/Infs
4. Fit XGBoost with balanced sample weights
5. Report macro-F1 on the held-out fold

In [ ]:
xgb_params = dict(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="mlogloss",
    n_jobs=-1,
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_f1 = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, y_train)):
    rng = np.random.default_rng(AUG_SEED + fold * 10_000)
    df_tr = train_df.iloc[tr_idx].reset_index(drop=True)
    df_va = train_df.iloc[va_idx].reset_index(drop=True)
    y_va  = le.transform(df_va["label"].values)

    df_tr_os = oversample_train_df(df_tr, OVERSAMPLE_MIN_PER_CLASS, rng)
    X_tr, y_tr = extract_features_parallel(df_tr_os, TRAIN_DIR, is_train=True,
                                            n_aug=N_AUG, seed=AUG_SEED + fold * 100)
    X_va, _    = extract_features_parallel(df_va, TRAIN_DIR, is_train=False,
                                            seed=AUG_SEED + fold * 200)

    scaler = StandardScaler()
    X_trs = np.nan_to_num(scaler.fit_transform(X_tr), nan=0.0, posinf=0.0, neginf=0.0)
    X_vas = np.nan_to_num(scaler.transform(X_va),     nan=0.0, posinf=0.0, neginf=0.0)

    sw  = compute_sample_weight("balanced", y_tr)
    clf = XGBClassifier(**xgb_params)
    clf.fit(X_trs, y_tr, sample_weight=sw)

    pred = clf.predict(X_vas)
    f1   = f1_score(y_va, pred, average="macro")
    fold_f1.append(f1)
    print(f"Fold {fold}: macro-F1 = {f1:.4f}  |  train samples = {X_trs.shape[0]}")

print(f"\nCV macro-F1: {np.mean(fold_f1):.4f} ± {np.std(fold_f1):.4f}")
print("Per-fold:", [round(x, 4) for x in fold_f1])

### 1.8 Full Training and Test Submission

In [ ]:
rng_full  = np.random.default_rng(AUG_SEED + 99_999)
df_full_os = oversample_train_df(train_df, OVERSAMPLE_MIN_PER_CLASS, rng_full)

X_full, y_full = extract_features_parallel(df_full_os, TRAIN_DIR, is_train=True,
                                            n_aug=N_AUG, seed=AUG_SEED)

scaler_full = StandardScaler()
X_full_s = np.nan_to_num(scaler_full.fit_transform(X_full), nan=0.0, posinf=0.0, neginf=0.0)

sw_full   = compute_sample_weight("balanced", y_full)
final_clf = XGBClassifier(**xgb_params)
final_clf.fit(X_full_s, y_full, sample_weight=sw_full)

X_test, _ = extract_features_parallel(test_df, TEST_DIR, is_train=False)
X_test_s  = np.nan_to_num(scaler_full.transform(X_test), nan=0.0, posinf=0.0, neginf=0.0)

y_pred     = final_clf.predict(X_test_s)
labels_out = le.inverse_transform(y_pred)

sub = pd.DataFrame({"ID": test_df["ID"], "label": labels_out})
sub.to_csv(f"submission_xgboost_aug{N_AUG}.csv", index=False)
print("Saved submission_xgboost_aug{}.csv".format(N_AUG))
print(sub["label"].value_counts())

---
## Part 2 — Deep Learning Pipeline

### 2.1 Preprocessing: Offline WBC Crop

Before training CNNs, we save cropped images to disk (organized by class for `ImageFolder` compatibility). This avoids re-running the crop at every epoch. EfficientNet uses 300×300 input; ConvNeXt uses 224×224.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import autocast, GradScaler
from torchvision import transforms, datasets, models
from torchvision.transforms import RandAugment
from torch.utils.data import DataLoader, WeightedRandomSampler, Dataset
import torchvision.transforms.functional as TF
import timm
from sklearn.model_selection import StratifiedKFold
import datetime

In [ ]:
def prepare_dataset(csv_path, src_dir, dst_dir, is_test=False):
    """Crop and save images. Training images go into class subdirectories."""
    df = pd.read_csv(csv_path)
    os.makedirs(dst_dir, exist_ok=True)
    for _, row in tqdm(df.iterrows(), total=len(df)):
        img = np.array(Image.open(os.path.join(src_dir, row['ID'])).convert("RGB"))
        cropped = apply_wbc_crop_robust(img)
        if not is_test:
            class_dir = os.path.join(dst_dir, str(row['label']))
            os.makedirs(class_dir, exist_ok=True)
            save_path = os.path.join(class_dir, row['ID'])
        else:
            save_path = os.path.join(dst_dir, row['ID'])
        Image.fromarray(cropped).save(save_path, quality=95)

# Run once to generate cropped datasets
# prepare_dataset("train_metadata.csv", "./train", "./data_cropped/train")
# prepare_dataset("test_metadata.csv",  "./test",  "./data_cropped/test",  is_test=True)

# For EfficientNet/ConvNeXt (same crop, different resize at load time)
# prepare_dataset("train_metadata.csv", "./train", "./data_cropped/train_eff")
# prepare_dataset("test_metadata.csv",  "./test",  "./data_cropped/test_eff", is_test=True)

### 2.2 Dataset with Class-Conditional Augmentation

Majority classes (BA, BL, EO, LY, MO, MY, SNE) get standard flips + small rotation + mild ColorJitter. Minority classes (BNE, MMY, PC, PLY, PMY, VLY) get RandAugment instead, which searches a broader set of transformations. Both paths use a `WeightedRandomSampler` to balance the effective class distribution per batch.

In [ ]:
# Class indices for the 6 most under-represented cell types
MINORITY_CLASSES = [2, 5, 8, 9, 10, 12]  # BNE, MMY, PC, PLY, PMY, VLY


class MinorityAugDataset(Dataset):
    """Applies strong augmentation to minority classes and base augmentation to the rest."""

    def __init__(self, subset, base_transform, strong_transform):
        self.subset           = subset
        self.base_transform   = base_transform
        self.strong_transform = strong_transform

    def __getitem__(self, index):
        img, label = self.subset[index]
        t = self.strong_transform if label in MINORITY_CLASSES else self.base_transform
        return t(img), label

    def __len__(self):
        return len(self.subset)


def _make_transforms(img_size):
    base = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    strong = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        RandAugment(num_ops=2, magnitude=9),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    val = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    return base, strong, val


def _weighted_sampler(targets):
    counts = np.array([len(np.where(targets == t)[0]) for t in np.unique(targets)])
    w      = 1.0 / counts
    sample_weights = torch.from_numpy(np.array([w[t] for t in targets])).double()
    return WeightedRandomSampler(sample_weights, len(sample_weights))


def get_loaders_fold(data_dir, fold_idx, n_folds=5, batch_size=64, img_size=300):
    full_dataset = datasets.ImageFolder(data_dir, transform=None)
    targets      = np.array(full_dataset.targets)

    skf    = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    splits = list(skf.split(np.zeros(len(targets)), targets))
    train_indices, val_indices = splits[fold_idx]

    base, strong, val = _make_transforms(img_size)

    train_dataset = MinorityAugDataset(
        torch.utils.data.Subset(full_dataset, train_indices), base, strong)
    val_dataset   = MinorityAugDataset(
        torch.utils.data.Subset(full_dataset, val_indices),   val,  val)

    sampler = _weighted_sampler(targets[train_indices])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler,   num_workers=4)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, num_workers=4)
    return train_loader, val_loader, full_dataset.classes

### 2.3 Model Architectures

- **ResNet-50**: ImageNet pretrained, final FC replaced with Dropout(0.5) + Linear(13)
- **EfficientNet-B3**: ImageNet pretrained (native 300×300), classifier head replaced similarly
- **ConvNeXt-Large**: loaded via `timm` (native 224×224), `num_classes=13` set directly

In [ ]:
def get_model(num_classes=13, model_name='resnet50'):
    if model_name == 'resnet50':
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        model.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(model.fc.in_features, num_classes),
        )
    elif model_name == 'efficientnet_b3':
        model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
        model.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(model.classifier[1].in_features, num_classes),
        )
    else:
        raise ValueError(f"Unknown model: {model_name}")
    return model


def get_convnext(num_classes=13):
    return timm.create_model('convnext_large', pretrained=True, num_classes=num_classes)

### 2.4 Focal Loss and Mixup

Focal Loss down-weights easy examples and focuses learning on hard or rare ones — useful here given the extreme imbalance. We use square-root class weights to smooth the gradient rather than raw inverse counts.

Mixup linearly interpolates pairs of images and their labels, which improves calibration on rare classes.

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, label_smoothing=0.1):
        super().__init__()
        self.alpha           = alpha
        self.gamma           = gamma
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets,
                             weight=self.alpha,
                             label_smoothing=self.label_smoothing,
                             reduction='none')
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()


def mixup_data(x, y, alpha=0.2):
    lam   = np.random.beta(alpha, alpha) if alpha > 0 else 1
    index = torch.randperm(x.size(0)).to(x.device)
    return lam * x + (1 - lam) * x[index], y, y[index], lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

### 2.5 Training Loop — ResNet-50 (baseline)

Single 80/20 split, weighted cross-entropy with square-root smoothing, `ReduceLROnPlateau` on macro-F1, saves the best checkpoint.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Square-root smoothed class weights
counts  = torch.tensor([415, 2012, 391, 861, 8101, 360, 2746, 441, 68, 11, 114, 13015, 366], dtype=torch.float)
weights = 1.0 / torch.sqrt(counts)
weights = (weights / weights.sum() * len(counts)).to(device)
CLASS_NAMES = ['BA', 'BL', 'BNE', 'EO', 'LY', 'MMY', 'MO', 'MY', 'PC', 'PLY', 'PMY', 'SNE', 'VLY']


def train_resnet50(data_dir="./data_cropped/train", n_epochs=50, batch_size=32, img_size=256):
    full_dataset = datasets.ImageFolder(data_dir, transform=None)
    class_names  = full_dataset.classes
    n = len(full_dataset)
    train_size, val_size = int(0.8 * n), n - int(0.8 * n)

    base, _, val_t = _make_transforms(img_size)
    train_idx, val_idx = torch.utils.data.random_split(
        range(n), [train_size, val_size], generator=torch.Generator().manual_seed(42))

    train_dataset = MinorityAugDataset(torch.utils.data.Subset(full_dataset, train_idx), base, base)
    val_dataset   = MinorityAugDataset(torch.utils.data.Subset(full_dataset, val_idx),   val_t, val_t)
    sampler       = _weighted_sampler([full_dataset.targets[i] for i in train_idx])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler, num_workers=4)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False,   num_workers=4)

    model     = get_model(num_classes=len(class_names), model_name='resnet50').to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'max', patience=2, factor=0.5)

    best_f1 = 0.0
    for epoch in range(n_epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch} [train]"):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(inputs), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch} [val]"):
                preds = torch.argmax(model(inputs.to(device)), 1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.numpy())

        macro_f1 = f1_score(all_labels, all_preds, average='macro')
        print(f"Epoch {epoch:3d} | train loss {running_loss/len(train_loader):.4f} "
              f"| macro-F1 {macro_f1:.4f} | lr {optimizer.param_groups[0]['lr']:.2e}")
        print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))

        scheduler.step(macro_f1)
        if macro_f1 > best_f1:
            best_f1 = macro_f1
            torch.save(model.state_dict(), "best_wbc_model.pth")
            print(f"  -> checkpoint saved (macro-F1={best_f1:.4f})")


# train_resnet50()

### 2.6 Training Loop — EfficientNet-B3 / ConvNeXt-Large with K-Fold

Key differences from the ResNet baseline:
- **Focal Loss** instead of cross-entropy
- **Automatic Mixed Precision** (AMP) to halve VRAM usage for ConvNeXt-Large
- **5-fold stratified cross-validation** — each fold saves its own checkpoint (`best_efficientnet_fold{k}.pth` / `best_convnext_fold{k}.pth`)
- EfficientNet: lr=1e-4, input 300×300; ConvNeXt: lr=5e-5, input 224×224

In [ ]:
def log_message(message, log_file="training.log"):
    ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    msg = f"[{ts}] {message}"
    print(msg)
    with open(log_file, "a") as f:
        f.write(msg + "\n")


def train_kfold(use_convnext=False, n_epochs=60, n_folds=5):
    if use_convnext:
        arch       = "convnext"
        img_size   = 224
        batch_size = 32
        lr         = 5e-5
        data_dir   = "./data_cropped/train_eff"
    else:
        arch       = "efficientnet"
        img_size   = 300
        batch_size = 64
        lr         = 1e-4
        data_dir   = "./data_cropped/train_eff"

    criterion = FocalLoss(alpha=None, gamma=2.0, label_smoothing=0.1)

    for fold in range(n_folds):
        log_message(f"{'='*50}\n  {arch.upper()} — Fold {fold+1}/{n_folds}\n{'='*50}")

        train_loader, val_loader, class_names = get_loaders_fold(
            data_dir, fold_idx=fold, n_folds=n_folds,
            batch_size=batch_size, img_size=img_size)

        model = (
            get_convnext(num_classes=len(class_names)) if use_convnext
            else get_model(num_classes=len(class_names), model_name='efficientnet_b3')
        ).to(device)

        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'max', patience=5, factor=0.5)
        scaler    = GradScaler('cuda')

        best_f1       = 0.0
        ckpt_path     = f"best_{arch}_fold{fold}.pth"
        cm_save_dir   = f"./CNN_models_{arch}/confusion_matrix_{arch}/fold{fold}"
        os.makedirs(cm_save_dir, exist_ok=True)

        for epoch in range(n_epochs):
            model.train()
            running_loss = 0.0
            for inputs, labels in tqdm(train_loader, desc=f"Fold {fold} Epoch {epoch} [train]"):
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                with autocast('cuda'):
                    loss = criterion(model(inputs), labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                running_loss += loss.item()

            model.eval()
            all_preds, all_labels = [], []
            with torch.no_grad():
                for inputs, labels in tqdm(val_loader, desc=f"Fold {fold} Epoch {epoch} [val]"):
                    preds = torch.argmax(model(inputs.to(device)), 1)
                    all_preds.extend(preds.cpu().numpy())
                    all_labels.extend(labels.numpy())

            macro_f1 = f1_score(all_labels, all_preds, average='macro')
            log_message(
                f"Epoch {epoch} | Train Loss: {running_loss/len(train_loader):.4f} "
                f"| Macro-F1: {macro_f1:.4f} | LR: {optimizer.param_groups[0]['lr']:.2e}"
            )
            print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))

            # Save confusion matrix
            cm = confusion_matrix(all_labels, all_preds)
            plt.figure(figsize=(12, 10))
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                        xticklabels=class_names, yticklabels=class_names)
            plt.xlabel('Predicted'); plt.ylabel('True')
            plt.title(f'Fold {fold} Epoch {epoch}')
            plt.savefig(f"{cm_save_dir}/confusion_matrix_epoch_{epoch}.png")
            plt.close()

            scheduler.step(macro_f1)
            if macro_f1 > best_f1:
                best_f1 = macro_f1
                torch.save(model.state_dict(), ckpt_path)
                log_message(f"  -> checkpoint saved: {ckpt_path}  (macro-F1={best_f1:.4f})")


# Train EfficientNet-B3 across 5 folds:
# train_kfold(use_convnext=False)

# Train ConvNeXt-Large across 5 folds:
# train_kfold(use_convnext=True)

### 2.7 Inference — ResNet-50 with 4-View TTA

At test time we average softmax probabilities over 4 augmented views: original, horizontal flip, vertical flip, and both flips combined.

In [ ]:
def predict_resnet50(model_path="best_wbc_model.pth",
                     test_dir="./data_cropped/test",
                     img_size=256,
                     out_csv="submission_resnet50_tta.csv"):
    test_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    model = get_model(num_classes=len(CLASS_NAMES), model_name='resnet50')
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device).eval()

    test_df_   = pd.read_csv("test_metadata.csv")
    predictions = []

    with torch.no_grad():
        for _, row in tqdm(test_df_.iterrows(), total=len(test_df_)):
            image = Image.open(os.path.join(test_dir, row['ID'])).convert("RGB")
            views = [image, TF.hflip(image), TF.vflip(image), TF.hflip(TF.vflip(image))]
            batch = torch.stack([test_transform(v) for v in views]).to(device)
            probs = torch.softmax(model(batch), dim=1).mean(dim=0)
            predictions.append(CLASS_NAMES[probs.argmax().item()])

    test_df_['label'] = predictions
    test_df_[['ID', 'label']].to_csv(out_csv, index=False)
    print(f"Saved: {out_csv}")


# predict_resnet50()

### 2.8 Inference — Ensemble (EfficientNet × 5 + ConvNeXt × 5) with 8-View TTA

Each of the 10 fold checkpoints independently produces an 8-view TTA prediction (4 rotation angles × original + horizontal flip). Per-model logits are averaged before softmax, then the 10 probability vectors are averaged across models. Optionally, we weight each model by its validation macro-F1 (via softmax-weighted combination).

In [ ]:
def tta_predict_single(model, image, transform):
    """8-view TTA: 4 rotation angles x {original, h-flip}. Returns softmax prob vector (13,)."""
    logits = []
    for angle in [0, 90, 180, 270]:
        img_rot = TF.rotate(image, angle)
        logits.append(model(transform(img_rot).unsqueeze(0).to(device)))
        logits.append(model(transform(TF.hflip(img_rot)).unsqueeze(0).to(device)))
    avg_logits = torch.mean(torch.cat(logits, dim=0), dim=0)
    return F.softmax(avg_logits, dim=0)


def get_transform(img_size):
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])


# Validation macro-F1 per fold, used for weighted ensemble
EFFNET_F1   = [0.6871, 0.6414, 0.6295, 0.6109, 0.6419]
CONVNEXT_F1 = [0.6253, 0.6656, 0.6505, 0.6288, 0.6910]


def softmax_weights(f1_scores, temperature=0.5):
    """Convert F1 scores to normalized ensemble weights."""
    f1  = np.array(f1_scores)
    exp = np.exp(f1 / temperature)
    return exp / exp.sum()


def load_ensemble_models(weighted=True):
    all_f1 = EFFNET_F1 + CONVNEXT_F1
    w      = softmax_weights(all_f1) if weighted else np.ones(10) / 10

    configs = (
        [(get_model(13, 'efficientnet_b3'), f"best_efficientnet_fold{k}.pth", 300) for k in range(5)] +
        [(timm.create_model('convnext_large', pretrained=False, num_classes=13),
          f"best_convnext_fold{k}.pth", 224) for k in range(5)]
    )
    loaded = []
    for i, (model, ckpt, img_size) in enumerate(configs):
        model.load_state_dict(torch.load(ckpt, map_location=device))
        model.to(device).eval()
        loaded.append((model, get_transform(img_size), w[i]))
        print(f"Loaded {ckpt}  weight={w[i]:.4f}")
    return loaded


def predict_ensemble(test_dir="./data_cropped/test_eff",
                     out_csv="submission_ensemble.csv",
                     weighted=True):
    models_  = load_ensemble_models(weighted=weighted)
    test_df_ = pd.read_csv("test_metadata.csv")
    predictions = []

    with torch.no_grad():
        for _, row in tqdm(test_df_.iterrows(), total=len(test_df_)):
            image = Image.open(os.path.join(test_dir, row['ID'])).convert("RGB")
            weighted_prob = torch.zeros(13).to(device)
            for model, transform, weight in models_:
                weighted_prob += weight * tta_predict_single(model, image, transform)
            predictions.append(CLASS_NAMES[weighted_prob.argmax().item()])

    test_df_['label'] = predictions
    test_df_[['ID', 'label']].to_csv(out_csv, index=False)
    print(f"Saved: {out_csv}")


# predict_ensemble(out_csv="submission_ensemble_10model_tta8.csv")

### 2.9 Post-processing (Optional)

The largest source of confusion is BNE ↔ SNE: SNE has 33× more samples. When the model predicts BNE with low confidence and SNE is a plausible alternative, we boost SNE's probability. A similar rule applies to VLY ↔ LY.

In [ ]:
def post_process(avg_prob):
    prob = avg_prob.clone()
    pred_idx = prob.argmax().item()

    # BNE (idx=2) vs SNE (idx=11): SNE has 33x more training samples
    if pred_idx == 2:
        if prob[2].item() < 0.45 and prob[11].item() > 0.15:
            prob[11] *= 1.5
            pred_idx = prob.argmax().item()

    # VLY (idx=12) vs LY (idx=4): LY has 22x more training samples
    if pred_idx == 12:
        if prob[12].item() < 0.40 and prob[4].item() > 0.20:
            prob[4] *= 1.3
            pred_idx = prob.argmax().item()

    return pred_idx


def predict_ensemble_with_postprocess(test_dir="./data_cropped/test_eff",
                                      out_csv="submission_ensemble_pp.csv"):
    models_  = load_ensemble_models(weighted=True)
    test_df_ = pd.read_csv("test_metadata.csv")
    predictions = []

    with torch.no_grad():
        for _, row in tqdm(test_df_.iterrows(), total=len(test_df_)):
            image = Image.open(os.path.join(test_dir, row['ID'])).convert("RGB")
            weighted_prob = torch.zeros(13).to(device)
            for model, transform, weight in models_:
                weighted_prob += weight * tta_predict_single(model, image, transform)
            predictions.append(CLASS_NAMES[post_process(weighted_prob)])

    test_df_['label'] = predictions
    test_df_[['ID', 'label']].to_csv(out_csv, index=False)
    print(f"Saved: {out_csv}")


# predict_ensemble_with_postprocess()